# OCR-Aware Image Region Suppression

Explore how optional OCR text boxes influence image-region candidate decisions. This notebook does not run OCR by default; it uses manual/fake boxes so the geometry layer can be reviewed without Tesseract, EasyOCR, or model downloads.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

from utils.image.layout.analyzer import LayoutAnalyzer
from utils.text.geometry import OCRTextBox, build_text_mask

REPO_ROOT = Path.cwd()
OUTPUT_DIR = REPO_ROOT / "data" / "notebook_outputs" / "ocr_aware_suppression"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Synthetic Page With Text And Diagram

In [ ]:
def synthetic_page():
    image = np.full((520, 640), 255, dtype=np.uint8)
    text_boxes = []
    for i, y in enumerate(range(40, 240, 24), start=1):
        cv2.line(image, (40, y), (390, y), 0, 2)
        text_boxes.append(OCRTextBox(text=f"line {i}", bbox=(35, y - 9, 365, 16), confidence=0.94, level="line"))
    cv2.rectangle(image, (420, 80), (585, 255), 0, 3)
    cv2.arrowedLine(image, (440, 220), (565, 105), 0, 3)
    cv2.putText(image, "A", (455, 130), cv2.FONT_HERSHEY_SIMPLEX, 0.9, 0, 2)
    text_boxes.append(OCRTextBox(text="A", bbox=(450, 105, 35, 38), confidence=0.90, level="word"))
    return image, text_boxes

page, ocr_boxes = synthetic_page()
plt.figure(figsize=(10, 8))
plt.imshow(page, cmap="gray")
plt.axis("off")
plt.show()

## Visualize OCR Text Mask

In [ ]:
mask = build_text_mask(page.shape[:2], ocr_boxes, dilation_px=4)
overlay = cv2.cvtColor(page, cv2.COLOR_GRAY2RGB)
overlay[mask > 0] = (255, 180, 180)
plt.figure(figsize=(10, 8))
plt.imshow(overlay)
plt.title("OCR text mask overlay")
plt.axis("off")
plt.show()

## Run Detector With And Without OCR Boxes

In [ ]:
analyzer = LayoutAnalyzer({
    "enabled_detectors": ["figure", "contours"],
    "contours_always": True,
    "filter_text_like": True,
})

without_ocr = analyzer.detect_image_regions_with_diagnostics(page)
with_ocr = analyzer.detect_image_regions_with_diagnostics(page, ocr_text_boxes=ocr_boxes)

print("Without OCR boxes:", len(without_ocr["accepted"]), "accepted /", len(without_ocr["rejected"]), "rejected")
print("With OCR boxes:", len(with_ocr["accepted"]), "accepted /", len(with_ocr["rejected"]), "rejected")
with_ocr["rejected"][:3]

In [ ]:
def draw_result(image, diagnostics, title):
    canvas = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    for item in diagnostics["rejected"]:
        region = item["region"]
        x, y, w, h = region["x"], region["y"], region["width"], region["height"]
        cv2.rectangle(canvas, (x, y), (x + w, y + h), (220, 80, 80), 2)
        cv2.putText(canvas, item.get("rejection_reason", "rejected"), (x, max(18, y - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (220, 80, 80), 1)
    for item in diagnostics["accepted"]:
        x, y, w, h = item["x"], item["y"], item["width"], item["height"]
        cv2.rectangle(canvas, (x, y), (x + w, y + h), (40, 160, 40), 3)
        reason = item.get("metadata", {}).get("accepted_reason", "accepted")
        cv2.putText(canvas, reason, (x, max(18, y - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (40, 160, 40), 1)
    plt.figure(figsize=(10, 8))
    plt.imshow(canvas)
    plt.title(title)
    plt.axis("off")
    plt.show()

draw_result(page, without_ocr, "Visual-only diagnostics")
draw_result(page, with_ocr, "OCR-aware diagnostics")

## Mixed-Region Refinement

Enable the optional OCR-mask refinement layer to inspect original versus refined bboxes. This remains disabled by default in runtime.

In [ ]:
refinement_analyzer = LayoutAnalyzer({
    "enabled_detectors": ["figure", "contours"],
    "contours_always": True,
    "filter_text_like": True,
    "region_enable_mixed_region_refinement": True,
    "region_mixed_region_min_ocr_overlap": 0.10,
})
with_refinement = refinement_analyzer.detect_image_regions_with_diagnostics(page, ocr_text_boxes=ocr_boxes)
print("Refinement events:")
with_refinement.get("refinement", [])

In [ ]:
draw_result(page, with_refinement, "OCR-aware diagnostics with mixed-region refinement")
for item in with_refinement["accepted"]:
    metadata = item.get("metadata", {})
    print(item["x"], item["y"], item["width"], item["height"], {
        "role": metadata.get("region_role"),
        "needs_review": metadata.get("needs_review"),
        "refinement_applied": metadata.get("refinement_applied"),
        "mixed_reason": metadata.get("mixed_reason"),
        "original_bbox": metadata.get("original_bbox"),
        "refined_bbox": metadata.get("refined_bbox"),
    })

## Real Page Review Hook

To use real pages, load a DFD or Corpus 2 image and provide OCR boxes from existing metadata or a small hand-authored list. If no OCR boxes are available, the detector intentionally falls back to visual-only behavior.